# Phase 3 — embedding extraction on Colab T4 (all-in-one)

Clones the repo, builds the **locked** environment, fetches and verifies `RQ.csv`, runs the GPU smoke test, a 200-example pilot, then the **full 4,997-example extraction** with automatic resume.

**Before running:** Runtime → Change runtime type → **T4 GPU**. Set `REPO_URL` in the next cell (push this repo to GitHub first; for a private repo use `https://<PAT>@github.com/<you>/rq-replication.git`). `uv sync` downloads the locked Linux CUDA torch + Qwen3-4B fp16 (~8 GB model, ~3 GB wheels) — a few minutes, once per runtime.

**Compatibility (checked 2026-11-04 against Colab's 2026-09-16 image update):**

| Colab change | Impact here | Verdict |
|---|---|---|
| Python 3.12.13 → **3.13.15** | `requires-python >=3.11`; `uv.lock` carries cp313 manylinux wheels for every package incl. torch 2.15 (verified in the lock file) | OK |
| Ubuntu 22.04 → 24.04 | no OS-specific code; the uv `.venv` is self-contained | OK |
| Preinstalled package bumps (pandas, tokenizers, jax, numba, …) | irrelevant — `uv sync` builds an isolated venv with locked versions; Colab's system packages are untouched | OK |
| Antigravity CLI pre-installed in Colab Terminal | orthogonal: this notebook needs no CLI; the [runbook](../docs/COLAB_RUNBOOK.md) CLI flow drives Colab from *your* terminal | OK |
| T4 GPU image | T4 has **no bf16** → config already uses fp16; Qwen3-4B fp16 ≈ 8.1 GB < 16 GB VRAM | OK |

One open risk: the locked torch 2.15 CUDA build vs the runtime's driver — the GPU assert + smoke test below catch that immediately; fallback documented in `docs/RISK_REGISTER.md` #7.

**If the runtime dies:** rerun cells 2–4, then jump straight to the full-extraction cell — it resumes from the partial checkpoint (in the same runtime session). Artifacts on the ephemeral disk vanish with the runtime → use the last cell to copy them to Drive.

In [ ]:
#@title 1. Configure + GPU check
REPO_URL = "https://github.com/gnaohaohk/rq-replication.git"  # @param {type:"string"}
RQ_CSV_URL = "https://raw.githubusercontent.com/ruyi101/rq-representation-probing/main/data/RQ.csv"
RQ_CSV_SHA256 = "316ccc067eea564b96ea90ef23a09cc769efb922bddce1255c6d968ae7a75236"  # canonical upstream blob (LF)
BATCH_SIZE = 16          # T4 pilot default; lower to 8/4 if OOM
PILOT_EXAMPLES = 200

import torch
assert torch.cuda.is_available(), "No GPU — select Runtime > Change runtime type > T4 GPU"
name = torch.cuda.get_device_name(0)
print(f"{name} | cc={torch.cuda.get_device_capability(0)} | bf16={torch.cuda.is_bf16_supported()} | torch {torch.__version__}")

In [ ]:
#@title 2. Clone repo + build the locked environment (~3–5 min)
import os, subprocess, time

def sh(cmd):
    t = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tail = (r.stdout or "")[-1500:] + ("\n" + r.stderr[-1500:] if r.returncode else "")
    print(tail, flush=True)
    if r.returncode:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    print(f"[{time.time() - t:.0f}s] ok: {cmd}", flush=True)

if not os.path.isdir("rq-replication"):
    sh(f"git clone {REPO_URL} rq-replication")
os.chdir("rq-replication")
sh("pip install -q uv")
sh("uv sync")  # locked env: CUDA torch for linux + cp313 per uv.lock

In [ ]:
#@title 3. Fetch RQ.csv (upstream blob) + verify checksum + validate
import hashlib, urllib.request

os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/RQ.csv"):
    urllib.request.urlretrieve(RQ_CSV_URL, "data/raw/RQ.csv")
h = hashlib.sha256(open("data/raw/RQ.csv", "rb").read()).hexdigest()
assert h == RQ_CSV_SHA256, f"RQ.csv checksum mismatch: got {h}"
print("RQ.csv checksum verified")
!uv run rq-probe validate --file data/raw/RQ.csv

In [ ]:
#@title 4. GPU smoke test (fast: extraction path on a tiny model + revision to pin)
!uv run python scripts/colab_smoke_test.py

In [ ]:
#@title 5. Pilot: 200 examples (downloads Qwen3-4B fp16 once, measures throughput + VRAM)
!uv run rq-probe extract --max-examples {PILOT_EXAMPLES} --batch-size {BATCH_SIZE}
import torch
print(f"\npeak VRAM: {torch.cuda.max_memory_allocated() / 2**30:.1f} GB / 16 GB")
print("scale the s/ex figure above by 4997 for the full-run estimate")

In [ ]:
#@title 6. Full extraction — 4,997 examples, resumable (rerun this cell after any interruption)
!uv run rq-probe extract --batch-size {BATCH_SIZE}

In [ ]:
#@title 7. Validate the artifact
import torch

p = "outputs/embeddings/RQ_question_with_context_Qwen3-4B_last_token.pt"
d = torch.load(p, map_location="cpu", weights_only=False)
e, meta = d["embeddings"], d["meta"]
assert meta["n_rows"] == 4997 and e.shape[0] == 4997, f"unexpected rows: {e.shape}"
assert d["ids"] == list(range(4997)), "row-alignment contract violated"
assert torch.isfinite(e).all(), "non-finite values in embeddings"
print(f"embeddings: {tuple(e.shape)} {e.dtype} | truncated rows: {meta['truncated_rows']}")
print({k: meta[k] for k in ("model_checkpoint", "pooling", "max_length", "dataset_sha256")})
print("~1.9 GB artifact ready — save it with the next cell before the runtime dies")

In [ ]:
#@title 8. (Optional) copy outputs to Google Drive
SAVE_TO_DRIVE = False  # @param {type:"boolean"}

if SAVE_TO_DRIVE:
    import datetime, shutil
    from google.colab import drive
    drive.mount("/content/drive")
    dst = f"/content/drive/MyDrive/rq-replication/{datetime.date.today().isoformat()}"
    shutil.copytree("outputs", dst, dirs_exist_ok=True)
    print("saved to", dst)
else:
    print("skipped — set SAVE_TO_DRIVE = True (or use `colab download` from the runbook)")